In [1]:
import json   
import pandas as pd
import numpy as np
from collections import defaultdict
from indicators import get_indicators_for_token
import os
from openpyxl import load_workbook, Workbook
import glob 
from tqdm import tqdm
from pathlib import Path
import gc
import multiprocessing as mp

In [2]:
base = Path("/app/")
candles_dir = base / "ML_Training_datasets" / "CandleData" / "Candles"
stats_dir = base / "ML_Training_datasets" / "CandleData" / "Stats"
output_dir = base / "ML_Training_datasets" / "Datasets" / "5S"
output_dir.mkdir(parents=True, exist_ok=True)

history_bars = 2500
prediction_horizons = (10, 20, 30, 40, 50)
timeframe_key = "5S"

In [3]:
SUM_COLS = [
    "buyCount",
    "sellCount",
    "buyVolumeSol",
    "sellVolumeSol"
]
MEAN_COLS = ["priceSol"]
WINDOW = 5

candle_tokens = {
    path.name.replace("_candles.json", "")
    for path in candles_dir.glob("*_candles.json")
}
stats_tokens = {
    path.name.replace("_stats.json", "")
    for path in stats_dir.glob("*_stats.json")
}
token_ids = sorted(candle_tokens & stats_tokens)

print(f"Found {len(token_ids)} tokens with both candle and stats files.")
token_ids[:20]

Found 39 tokens with both candle and stats files.


['34yXcY1QRF51RBLtXFmfM6pwaUwuv4x5x6rx6Fd1Pf3s',
 '3KFCgJ5R3zshW8hTDbzjSrrKSRYmKvsMfhc4Vo4iddxD',
 '46Y1ioz65PjY3hgbX95pDdQPZrwuTbpJRSEhEQVJcrsL',
 '4C47JJgDHztupNHeU9da7MroH676eXbPoUhEvx3FpfR3',
 '4cqyfY9pdAYxqxLsmnr9D9Gm8T83e9WjAMkq3pECywkN',
 '4mMDQ5kG9fFrBSQeedErsUoTBhY5KKnsKWGvenXRTwSy',
 '4xxM4cdb6MEsCxM52xvYqkNbzvdeWWsPDZrBcTqVGUar',
 '5imMdVjpsBM84z9JvYw1aHfNF8zTA3unrmWeYUQKkiQs',
 '6UYbX1x8YUcFj8YstPYiZByG7uQzAq2s46ZWphUMkjg5',
 '6oFWm7KPLfxnwMb3z5xwBoXNSPP3JJyirAPqPSiVcnsp',
 '8LTRgxZ2KWDc2sDGuYAe5VtDgguAzteSeQUJisLQ5BVA',
 '8WwcNqdZjCY5Pt7AkhupAFknV2txca9sq6YBkGzLbvdt',
 '96PNGF9SatBSLjc5MnoDYovEDANgocJXweJbrwC57Dga',
 '9B8xXGabWeEfCLzwHAwwbjZjQamiFsh7WC8Y5bkvPbqP',
 'A6KHMiFzn9AM7VKBtVP4fZNY9bCo2jP63R9dphaW1vrq',
 'AFaYrFH7hUYnkv5mbvbnRHwX29M9JzUL3ySGtQz69AUv',
 'AVs9TA4nWDzfPJE9gGVNJMVhcQy3V9PGazuz33BfG2RA',
 'AZThnSj6jrSDjFERxwK6jBQh4siLJxshnQTp4K7vUGq5',
 'BFn7n6xPSX4ZsLwBMfFvAaDKGvQbyDcwWAj8UGDiPHr6',
 'BLXGHhArwoPkwwnoJpuuWVDMWwbzJ24RmoSbj1JqSsST']

In [4]:
def flatten_and_align_indicators(ind_dict, N):
    """
    Normalize indicator outputs so downstream code can safely flatten them.

    - Scalar indicators stay scalar (do NOT expand to lists).
    - List indicators are left-padded/truncated to length N.
    - List[dict] indicators are split into separate list columns.
    """
    aligned = {}

    def pad_list(lst, length, fill=np.nan):
        if lst is None:
            lst = []
        lst = list(lst)
        if len(lst) < length:
            return [fill] * (length - len(lst)) + lst
        return lst[-length:]

    for key, val in ind_dict.items():
        if isinstance(val, list):
            if val and isinstance(val[0], dict):
                subkeys = set()
                for item in val:
                    if isinstance(item, dict):
                        subkeys.update(item.keys())
                for subkey in sorted(subkeys):
                    series = [
                        (item.get(subkey, np.nan) if isinstance(item, dict) else np.nan)
                        for item in val
                    ]
                    aligned[f"{key}_{subkey}"] = pad_list(series, N)
            else:
                aligned[key] = pad_list(val, N)
        else:
            aligned[key] = val

    return aligned

In [5]:
def list_to_rows(df, sequence_mode="latest", max_points=20):
    """
    Flatten indicator list columns without blowing up row-count.

    Parameters
    ----------
    df : pd.DataFrame
    sequence_mode : str
        - "latest": keep only the most recent value from each list column.
        - "window": keep the trailing `max_points` values as separate columns.
        - "explode": legacy behaviour (row explosion), kept only when explicitly needed.
    max_points : int
        Number of trailing points to keep when sequence_mode="window".

    Notes
    -----
    Default mode is "latest" so class distribution is preserved
    (one training row per label event) and memory stays bounded.
    """
    if df.empty:
        return df.copy()

    out = df.copy()
    list_cols = [col for col in out.columns if out[col].map(lambda x: isinstance(x, list)).any()]

    if not list_cols:
        return out.reset_index(drop=True)

    def normalize_list(v):
        if isinstance(v, list):
            return v
        if pd.isna(v):
            return []
        return [v]

    if sequence_mode == "latest":
        for col in list_cols:
            out[col] = out[col].map(lambda v: (normalize_list(v)[-1] if len(normalize_list(v)) else np.nan))
        return out.reset_index(drop=True)

    if sequence_mode == "window":
        max_points = max(1, int(max_points))
        for col in list_cols:
            seq = out[col].map(normalize_list)
            for lag in range(max_points):
                # lag=0 => latest value, lag=1 => previous, ...
                out[f"{col}_lag_{lag}"] = seq.map(
                    lambda s, lag=lag: s[-(lag + 1)] if len(s) > lag else np.nan
                )
            out.drop(columns=[col], inplace=True)
        return out.reset_index(drop=True)

    if sequence_mode == "explode":
        scalar_cols = [col for col in out.columns if col not in list_cols]

        lengths_df = pd.DataFrame(
            {
                col: out[col].map(lambda x: len(x) if isinstance(x, list) else 1)
                for col in list_cols
            },
            index=out.index,
        )
        max_len = lengths_df.max(axis=1).fillna(1).astype(int)

        def pad_list(val, length):
            seq = normalize_list(val)
            if len(seq) < length:
                return [np.nan] * (length - len(seq)) + seq
            return seq[-length:]

        list_part = pd.DataFrame(index=out.index)
        for col in list_cols:
            list_part[col] = [pad_list(v, l) for v, l in zip(out[col], max_len)]

        exploded = list_part.explode(list_cols, ignore_index=False)
        repeated_index = np.repeat(out.index.to_numpy(), max_len.to_numpy())
        scalar_part = out.loc[repeated_index, scalar_cols].reset_index(drop=True)

        exploded = exploded.reset_index(drop=True)
        return pd.concat([scalar_part, exploded], axis=1)

    raise ValueError("sequence_mode must be one of: 'latest', 'window', 'explode'")

In [6]:
def append_to_dataset(flattened_indicators, block_df, dataset, distance, extreme_type, pct_change, 
                      current_close, timeframe, high_wick, low_wick, window_size):
    """
    Enrich flattened indicators with additional features and append to dataset.
    Computes time_to_extreme, relative OHLC, volume stats, momentum, and time features.
    """

    # ---------- Time-to-extreme ----------
    flattened_indicators['time_to_extreme'] = distance * timeframe
    flattened_indicators['time_to_extreme_rel'] = distance / window_size  # relative to window length
    flattened_indicators['extreme_type'] = extreme_type
    flattened_indicators['change_percentage'] = pct_change

    # ---------- Wicks ----------
    flattened_indicators['high_wick'] = high_wick
    flattened_indicators['low_wick'] = low_wick

    # ---------- Volume features ----------
    vol = block_df['volume']
    flattened_indicators['volume_sum'] = vol.sum()
    flattened_indicators['volume_avg'] = vol.mean()
    flattened_indicators['volume_rel'] = vol.mean() / vol.iloc[0]
    flattened_indicators['volume_max_rel'] = vol.max() / vol.mean()

    # ---------- High/Low & Range ----------
    flattened_indicators['rel_high'] = block_df['high'].max() / current_close
    flattened_indicators['rel_low'] = block_df['low'].min() / current_close
    flattened_indicators['range_pct'] = (block_df['high'].max() - block_df['low'].min()) / current_close * 100
    flattened_indicators['subblock_len'] = distance

    # ---------- OHLC relative to first candle ----------
    first_candle = block_df.iloc[0]
    flattened_indicators['open_rel'] = first_candle['open'] / current_close
    flattened_indicators['high_rel'] = first_candle['high'] / current_close
    flattened_indicators['low_rel'] = first_candle['low'] / current_close
    flattened_indicators['close_rel'] = first_candle['close'] / current_close

    # ---------- Short-term momentum ----------
    if len(block_df) > 1:
        flattened_indicators['pct_change_1'] = (first_candle['close'] - first_candle['open']) / current_close * 100
        flattened_indicators['pct_change_2'] = (first_candle['close'] - block_df.iloc[1]['close']) / current_close * 100
    else:
        flattened_indicators['pct_change_1'] = 0.0
        flattened_indicators['pct_change_2'] = 0.0

    # ---------- Time features ----------
    ts = pd.to_datetime(first_candle['timestamp'])
    flattened_indicators['hour'] = ts.hour
    flattened_indicators['minute'] = ts.minute
    flattened_indicators['weekday'] = ts.weekday()
    flattened_indicators['timestamp'] = ts

    dataset.append(flattened_indicators)


In [7]:
def balance_memecoin_dataset_full(df, random_state=42):
    """
    Downsamples all classes in 'extreme_type' to match the smallest class size.
    Ensures equal representation of up, down, and stable.

    Parameters
    ----------
    df : pd.DataFrame
        Dataset containing 'extreme_type' column
    random_state : int
        Seed for reproducibility

    Returns
    -------
    pd.DataFrame
        Fully balanced dataset
    """
    counts = df['extreme_type'].value_counts()
    print("Before balancing:\n", counts)

    min_count = counts.min()
    dfs = []

    for label in counts.index:
        sampled = df[df['extreme_type'] == label].sample(n=min_count, random_state=random_state)
        dfs.append(sampled)

    df_balanced = pd.concat(dfs).sample(frac=1, random_state=random_state).reset_index(drop=True)

    print("\nAfter balancing:\n", df_balanced['extreme_type'].value_counts())
    return df_balanced


In [8]:
def add_common_features(base_flattened, subblock, current_close, ts_i): 
    base_flattened["volume_sum"] = subblock["volume"].sum() 
    base_flattened["volume_avg"] = subblock["volume"].mean() 
    base_flattened["range_pct"] = (subblock["high"].max() - subblock["low"].min()) / current_close * 100 
    first_candle = subblock.iloc[0] 
    base_flattened["open_rel"] = first_candle["open"] / current_close 
    base_flattened["high_rel"] = first_candle["high"] / current_close 
    base_flattened["low_rel"] = first_candle["low"] / current_close 
    base_flattened["close_rel"] = first_candle["close"] / current_close 
    base_flattened["hour"] = ts_i.hour 
    base_flattened["minute"] = ts_i.minute 
    base_flattened["weekday"] = ts_i.weekday() 
    base_flattened["timestamp"] = ts_i

In [9]:
def create_forward_label_dataset_clean(df, window_size, step_size,
                                       up_threshold, down_threshold,
                                       stable_threshold, timeframe,
                                       history_bars=300,
                                       target_samples=20,
                                       prediction_horizons=(10, 20, 30, 40, 50),
                                       get_indicators_for_token=get_indicators_for_token):
    """
    Build a time-series dataset by iterating once over each absolute candle index.

    - Start at the first candle that has at least `history_bars` prior candles.
    - Compute indicators using up to the most recent `history_bars` ending at that candle.
    - Create fixed-horizon targets for what happened after 10/20/30/40/50 candles.

    Notes
    -----
    `window_size`, `step_size`, `target_samples`, and the threshold arguments are kept
    in the signature for backward compatibility, but this version no longer samples
    inside overlapping windows or builds up/down/stable labels.
    """
    dataset = []

    df_closes = df["close"].values
    df_len = len(df)
    prediction_horizons = tuple(sorted(set(int(h) for h in prediction_horizons if int(h) > 0)))

    if not prediction_horizons:
        raise ValueError("prediction_horizons must contain at least one positive integer")

    max_horizon = max(prediction_horizons)
    start_idx = history_bars
    end_idx = df_len - max_horizon

    if end_idx <= start_idx:
        return pd.DataFrame(dataset)

    total_rows = end_idx - start_idx

    with tqdm(total=total_rows, desc="Rows") as pbar:

        for abs_i in range(start_idx, end_idx):
            current_close = df_closes[abs_i]
            ts_i = pd.to_datetime(df.iloc[abs_i]["timestamp"])

            # Build history subblock (no leakage) using at most the latest `history_bars` bars.
            start_sub = max(0, abs_i - (history_bars - 1))
            subblock = df.iloc[start_sub:abs_i + 1].copy()

            # Compute indicators once
            indicators = get_indicators_for_token(subblock.to_dict("records"))
            base_flattened = flatten_and_align_indicators(indicators, len(subblock))

            # Add common features once to base
            add_common_features(base_flattened, subblock, current_close, ts_i)

            row = base_flattened.copy()
            row["current_close"] = current_close

            for horizon in prediction_horizons:
                future_close = df_closes[abs_i + horizon]
                row[f"future_close_{horizon}"] = future_close
                row[f"future_return_pct_{horizon}"] = ((future_close - current_close) / current_close) * 100.0

            dataset.append(row)
            pbar.update(1)

    df_dataset = pd.DataFrame(dataset)
    df_dataset.fillna(np.nan, inplace=True)
    return df_dataset


In [10]:
def load_token_frame(token_id, timeframe_key="5S"):
    candles_file = candles_dir / f"{token_id}_candles.json"
    stats_file = stats_dir / f"{token_id}_stats.json"

    with candles_file.open("r", encoding="utf-8") as f:
        candle_data = json.load(f)

    with stats_file.open("r", encoding="utf-8") as f:
        stats_data = json.load(f)

    token_name = candle_data.get("name", token_id)
    candles = candle_data.get("timeframes", {}).get(timeframe_key, [])
    if not candles:
        raise ValueError(f"No {timeframe_key} candles found")

    df = pd.DataFrame(candles)
    if "timestamp" not in df.columns:
        raise ValueError("Candles data is missing 'timestamp'")

    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)

    stats_records = stats_data.get("stats", [])
    if not stats_records:
        raise ValueError("Stats data is empty")

    stats_df = pd.DataFrame(stats_data["stats"])
   
    required_stats_cols = ["createdAt", *SUM_COLS, *MEAN_COLS]
    missing_cols = [col for col in required_stats_cols if col not in stats_df.columns]
    if missing_cols:
        raise ValueError(f"Stats data is missing columns: {missing_cols}")

    stats_df["timestamp"] = pd.to_datetime(stats_df["createdAt"], utc=True)
    stats_df = stats_df.sort_values("timestamp")
    stats_df = stats_df.drop(columns=["createdAt"])

    stats_5m = stats_df.copy()
    stats_5m[SUM_COLS] = stats_df[SUM_COLS].rolling(WINDOW, min_periods=1).sum()
    stats_5m[MEAN_COLS] = stats_df[MEAN_COLS].rolling(WINDOW, min_periods=1).mean()
    stats_5m = stats_5m.rename(columns={c: f"last_5m_{c}" for c in SUM_COLS + MEAN_COLS})

    stats_5m["timestamp"] = pd.to_datetime(stats_5m["timestamp"], utc=True)
    stats_5m = stats_5m.sort_values("timestamp")

    df["timestamp"] = df["timestamp"].astype("datetime64[ns, UTC]")
    stats_5m["timestamp"] = stats_5m["timestamp"].astype("datetime64[ns, UTC]")

    df_final = pd.merge_asof(
        df,
        stats_5m,
        on="timestamp",
        direction="backward"
    )

    return token_name, df_final

In [ ]:
def process_single_token(job):
    file_number, token_id, timeframe_key, history_bars, prediction_horizons = job

    dataset = None
    exploded_dataset = None
    exploded_dataset_clean = None
    df_new = None
    df_final = None
    token_name = None

    try:
        token_name, df_final = load_token_frame(token_id, timeframe_key=timeframe_key)

        dataset = create_forward_label_dataset_clean(
            df=df_final,
            window_size=400,
            step_size=50,
            up_threshold=20.0,
            down_threshold=15.0,
            stable_threshold=10.0,
            timeframe=5,
            history_bars=history_bars,
            target_samples=100,
            prediction_horizons=prediction_horizons,
            get_indicators_for_token=get_indicators_for_token
        )

        if dataset.empty:
            return {
                "file_number": file_number,
                "token_id": token_id,
                "token_name": token_name,
                "status": "empty_dataset",
                "rows": 0,
                "output_file": None,
                "error": None,
            }

        exploded_dataset = list_to_rows(dataset, sequence_mode="latest")
        exploded_dataset_clean = exploded_dataset.dropna(how="any").reset_index(drop=True)

        if exploded_dataset_clean.empty:
            return {
                "file_number": file_number,
                "token_id": token_id,
                "token_name": token_name,
                "status": "empty_cleaned",
                "rows": 0,
                "output_file": None,
                "error": None,
            }

        df_new = exploded_dataset_clean.copy()
        df_new["token_id"] = token_id
        df_new["token_name"] = token_name

        out_path = output_dir / f"memecoin_training_dataset_{file_number}.csv"
        df_new.to_csv(out_path, index=False)

        return {
            "file_number": file_number,
            "token_id": token_id,
            "token_name": token_name,
            "status": "saved",
            "rows": int(len(df_new)),
            "output_file": out_path.name,
            "error": None,
        }

    except Exception as exc:
        return {
            "file_number": file_number,
            "token_id": token_id,
            "token_name": token_name,
            "status": "error",
            "rows": 0,
            "output_file": None,
            "error": str(exc),
        }

    finally:
        del dataset, exploded_dataset, exploded_dataset_clean, df_new, df_final, token_name
        gc.collect()


jobs = [
    (file_number, token_id, timeframe_key, history_bars, prediction_horizons)
    for file_number, token_id in enumerate(token_ids, start=1)
]

ctx = mp.get_context("spawn")
with ctx.Pool(processes=1, maxtasksperchild=1) as pool:
    for result in pool.imap(process_single_token, jobs):
        file_number = result["file_number"]
        token_id = result["token_id"]
        token_name = result["token_name"] or token_id

        if result["status"] == "saved":
            print(
                f"{file_number}, Processing token: {token_name} ({token_id})\n"
                f"  -> saved {result['rows']} rows to {result['output_file']}\n"
            )
        elif result["status"] == "empty_dataset":
            print(
                f"{file_number}, Processing token: {token_name} ({token_id})\n"
                "  -> dataset is empty, skipping output\n"
            )
        elif result["status"] == "empty_cleaned":
            print(
                f"{file_number}, Processing token: {token_name} ({token_id})\n"
                "  -> cleaned dataset is empty, skipping output\n"
            )
        else:
            print(f"{file_number}, Skipping token {token_id}: {result['error']}\n")

1, Processing token: Meep (34yXcY1QRF51RBLtXFmfM6pwaUwuv4x5x6rx6Fd1Pf3s)


Rows: 100% 14230/14230 [54:36<00:00,  4.34it/s]


  -> saved 14230 rows to memecoin_training_dataset_1.csv

2, Processing token: TripleT (3KFCgJ5R3zshW8hTDbzjSrrKSRYmKvsMfhc4Vo4iddxD)


Rows:   1% 922/66070 [01:12<1:29:41, 12.11it/s]